In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense
from sklearn import metrics
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import CategoricalNB
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
from sklearn.decomposition import PCA
from mlxtend.plotting import plot_decision_regions
from sklearn.tree import export_graphviz
from six import StringIO
from IPython.display import Image
import graphviz
%matplotlib inline


In [3]:
import dask.dataframe as dd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
import scipy.stats
from scipy.stats import chi2
%matplotlib inline
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV

Train and Test data scaling

In [4]:
import dask.array as da
df = pd.read_csv("/content/drive/MyDrive/new_train2.csv")

In [5]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [6]:
df_test = pd.read_csv("/content/drive/MyDrive/new_test2.csv")

In [7]:
df_test = df_test.loc[:, ~df_test.columns.str.contains('^Unnamed')]

In [8]:
df.head()

,TotLen Fwd Pkts,SYN Flag Cnt,Fwd Seg Size Min,Bwd IAT Mean,Pkt Len Min,Flow Byts/s,Bwd Pkts/s,Init Fwd Win Byts,CWE Flag Count,Label
0,49.0,0.0,8.0,0.000000,49.0,92972.501091,436.490615,-1.0,0.0,0
1,0.0,0.0,20.0,0.000000,0.0,0.000000,0.172838,8192.0,0.0,0
2,287.0,0.0,20.0,4502.333333,0.0,90431.436390,296.011248,65535.0,0.0,1
3,308.0,0.0,20.0,535.333333,0.0,772049.689400,2484.472050,65535.0,0.0,1
4,36.0,0.0,8.0,0.000000,36.0,97757.847534,896.860987,-1.0,0.0,0


In [9]:
y=df["Label"]
df = df.drop("Label",axis=1)
X=df

In [10]:
from sklearn.preprocessing import StandardScaler
std = StandardScaler()
X_t = std.fit_transform(X)
df_t = std.transform(df_test)

In [ ]:
X_t.shape

(8393080, 9)

NEURAL NETWORKS

In [11]:
from keras.models import Sequential
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.utils import np_utils
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder


In [12]:
X_t.shape

(8393080, 9)

In [13]:
from keras.models import Sequential
from keras.layers import Dense,Dropout
from keras.optimizers import SGD, Adam, Adadelta, RMSprop
import keras.backend as K

model = keras.Sequential()
model.add(Dense(28 , input_shape=(9,) , activation="relu" , name="Hidden_Layer_1"))
model.add(Dense(10 , activation="relu" , name="Hidden_Layer_2"))
model.add(Dense(1 , activation="sigmoid" , name="Output_Layer"))
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile( optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Hidden_Layer_1 (Dense)      (None, 28)                280       
                                                                 
 Hidden_Layer_2 (Dense)      (None, 10)                290       
                                                                 
 Output_Layer (Dense)        (None, 1)                 11        
                                                                 
Total params: 581
Trainable params: 581
Non-trainable params: 0
_________________________________________________________________


In [15]:
model.fit(X_t, y, epochs=10,batch_size=32)

Epoch 1/10
262284/262284 [==============================] - 487s 2ms/step - loss: 0.0938 - accuracy: 0.9783
Epoch 2/10
262284/262284 [==============================] - 488s 2ms/step - loss: 0.0824 - accuracy: 0.9828
Epoch 3/10
262284/262284 [==============================] - 478s 2ms/step - loss: 0.0790 - accuracy: 0.9840
Epoch 4/10
262284/262284 [==============================] - 474s 2ms/step - loss: 0.0781 - accuracy: 0.9841
Epoch 5/10
262284/262284 [==============================] - 474s 2ms/step - loss: 0.0768 - accuracy: 0.9845
Epoch 6/10
262284/262284 [==============================] - 476s 2ms/step - loss: 0.0751 - accuracy: 0.9846
Epoch 7/10
262284/262284 [==============================] - 475s 2ms/step - loss: 0.0750 - accuracy: 0.9848
Epoch 8/10
262284/262284 [==============================] - 475s 2ms/step - loss: 0.0750 - accuracy: 0.9847
Epoch 9/10
262284/262284 [==============================] - 475s 2ms/step - loss: 0.0772 - accuracy: 0.9848
Epoch 10/10
262284/262284 [=

PREDICTION

In [16]:
from torch import nn
logits = model(df_t)


In [17]:
logits

<tf.Tensor: shape=(1000000, 1), dtype=float32, numpy=
array([[7.5383000e-03],
       [7.5383000e-03],
       [7.5383000e-03],
       ...,
       [1.6860306e-06],
       [1.1335526e-02],
       [9.2683723e-03]], dtype=float32)>

In [18]:
logits.shape

TensorShape([1000000, 1])

In [20]:
ypred = (model.predict(df_t) > 0.5).astype("int32")

31250/31250 [==============================] - 38s 1ms/step


In [21]:
predd_lr2 = pd.DataFrame(ypred, columns =["Label"])
predd_lr2.to_csv('Pred_nn5.csv', index=True)  